## 1. Conexão

> **Conexão protegida. É proibido alterar, substituir ou excluir as células de conexão.**


O estudo usa a sessão Spark existente, o DB2 V3 para leitura nativa e o HDFS V1 para a publicação dos arquivos.


In [ ]:
from traceback import format_exc

try:
    from src.utils.gerenciador_local_v2 import GerenciadorLocal

    gerenciador_local = GerenciadorLocal(
        nome_sessao="radar-todos-1",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_local.criar_sessao_spark(
        db2=True,
        driver_memory="16g",
        num_executors=6,
        executor_memory="4g",
        executor_cores=4,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "2g",

            "spark.dynamicAllocation.enabled": "true",
            "spark.dynamicAllocation.minExecutors": "4",
            "spark.dynamicAllocation.initialExecutors": "6",
            "spark.dynamicAllocation.maxExecutors": "8",

            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",

            "spark.speculation": "false",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
            "spark.sql.adaptive.advisoryPartitionSizeInBytes": "64m",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "320",

            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    try:
        widget_cls = __import__("ipywidgets").Widget
        ipython = get_ipython()
        if ipython is not None:
            ipython.display_formatter.formatters["text/plain"].for_type(
                widget_cls,
                lambda *a, **k: None,
            )
    except (ImportError, NameError, AttributeError):
        pass

    print("Sessão Spark inicializada.")

except Exception as exc:
    print(type(exc).__name__)
    print(str(exc))
    print(format_exc())
    raise


In [ ]:
%run ./src/utils/gerenciador_spark_v2.ipynb
%run ./src/utils/gerenciador_db2_spark_v3.ipynb
%run ./src/utils/gerenciador_hdfs_spark_v1.ipynb


## Parâmetros do estudo GFP

Use HOMOLOGACAO para validar a infraestrutura e o download. AMOSTRAL exige uma lista ou arquivo local autorizado com um MCI por linha. CENSITARIO exige autorização explícita de carga. AUDITORIA compara as leituras originais e otimizadas no mesmo recorte. Configure as datas no intervalo [início, fim). A configuração inicial cobre um dia. Amplie progressivamente até sete dias após medir o desempenho.


In [ ]:
revisao_notebook = "2026-11-lotes-gfp"
modo = "HOMOLOGACAO"
data_inicial = "2026-11-24"
data_final_exclusiva = "2026-11-25"
consultas_selecionadas = ("Q00", "Q01", "Q01B", "Q02", "Q03", "Q04", "Q05", "Q06", "Q07", "Q08", "Q09", "Q10", "Q11")
mcis_autorizados = []
arquivo_populacao = None  # Arquivo UTF-8 no JupyterLab, um MCI por linha.
tamanho_amostra = None
semente = 42
tamanho_lote = 10
particoes = []  # Segmentos da tabela principal. Vazio usa todo o recorte solicitado.
experimentar_formula_particao = False
autorizar_censo = False
timeout_transacoes_segundos = 300
timeout_cadastros_segundos = 60
orcamento_global_segundos = 900
retomar_execucao = None  # Identificador exibido na execução anterior.
baixar_zip_jupyterlab = True
cluster_bbmagic = "CDP"
destino_hdfs = None


## Executar e publicar

Cada operação JDBC usa uma conexão de leitura independente. Unidades validadas são recuperáveis no HDFS. O ZIP informa a cobertura de cada resultado e o estado funcional da execução. O download automático confere o SHA-256 da publicação.


In [ ]:
from pathlib import Path
from uuid import uuid4

id_execucao_gfp = uuid4().hex
fonte_modulo = Path("src/utils/estudo_indicadores_gfp.py").read_text(encoding="utf-8")
estudo = {}
exec(compile(fonte_modulo, "estudo_indicadores_gfp.py", "exec"), estudo)
configuracao_estudo = {
    "modo": modo, "inicio": data_inicial, "fim": data_final_exclusiva,
    "semente": semente, "tamanho_amostra": tamanho_amostra,
    "fonte": "arquivo autorizado" if arquivo_populacao else "lista explícita" if mcis_autorizados else "sem lista",
    "consultas": list(consultas_selecionadas), "lote": tamanho_lote, "particoes": list(particoes),
    "experimental": experimentar_formula_particao, "autorizar_censo": autorizar_censo,
    "orcamento": orcamento_global_segundos, "retomar": retomar_execucao, "destino": destino_hdfs,
    "timeout_transacoes": timeout_transacoes_segundos, "timeout_cadastros": timeout_cadastros_segundos,
}
populacao_gfp = estudo["selecionar_populacao"](mcis_autorizados, arquivo_populacao, tamanho_amostra, semente)
estudo["validar_config"](configuracao_estudo, len(populacao_gfp))
consultas_estudo, plano_estudo = estudo["preparar_plano"](Path("03_CONSULTAS"), configuracao_estudo)
fontes_estudo = {
    "ESTUDO_RADAR.ipynb": estudo["preparar_notebook"](Path.cwd(), revisao_notebook),
    "estudo_indicadores_gfp.py": fonte_modulo,
}
print(modo, data_inicial, data_final_exclusiva, len(populacao_gfp), "clientes selecionados")


In [ ]:
from tempfile import TemporaryDirectory
from urllib.parse import urlsplit

ipython = get_ipython()
pasta_entrada_gfp = None
if populacao_gfp:
    from bbmagic import Hdfs

    # O código enviado ao Livy recebe apenas o caminho da pasta privada de transferência.
    entrada_gfp_local = None
    ipython.run_cell_magic(
        "spark", "-o entrada_gfp_local -n 1",
        f"pasta_entrada_gfp = criar_gerenciador_hdfs_spark().criar_diretorio('entrada_gfp_{id_execucao_gfp}')\n"
        "entrada_gfp_local = spark.createDataFrame([(pasta_entrada_gfp,)], 'pasta string')",
    )
    pasta_entrada_gfp = entrada_gfp_local.iloc[0]["pasta"]
    uri_entrada = urlsplit(pasta_entrada_gfp)
    if uri_entrada.scheme != "hdfs" or not uri_entrada.netloc or uri_entrada.path.rsplit("/", 1)[-1] != "entrada_gfp_" + id_execucao_gfp:
        raise RuntimeError("A pasta privada de transferência não corresponde à execução.")
    with TemporaryDirectory() as temp:
        local = Path(temp) / "entrada"
        with local.open("x", encoding="ascii") as saida:
            for mci in populacao_gfp:
                saida.write(f"{mci}\n")
        try:
            Hdfs(gerenciador_local.keytab, cluster=cluster_bbmagic).upload(
                hdfs_path=urlsplit(pasta_entrada_gfp).path + "/entrada", local_path=str(local), overwrite=False,
            )
        except Exception:
            ipython.run_cell_magic("spark", "", f"criar_gerenciador_hdfs_spark().remover({pasta_entrada_gfp!r}, recursivo=True)")
            raise RuntimeError("Falha na transferência privada da população autorizada.") from None

codigo_remoto_estudo = (
    f"recibo_zip_gfp = {{'execucao': {id_execucao_gfp!r}}}\n"
    f"fontes_gfp = {fontes_estudo!r}\n"
    "api_gfp = {}\n"
    "exec(compile(fontes_gfp['estudo_indicadores_gfp.py'], 'estudo_indicadores_gfp.py', 'exec'), api_gfp)\n"
    f"clientes_gfp = api_gfp['carregar_populacao_driver'](criar_gerenciador_hdfs_spark, {pasta_entrada_gfp!r})\n"
    f"api_gfp['executar_estudo'](criar_conector_db2_spark, {consultas_estudo!r}, {configuracao_estudo!r}, "
    f"fontes_gfp, criar_gerenciador_hdfs_spark, recibo_zip_gfp, plano={plano_estudo!r}, clientes=clientes_gfp, sessao_spark=spark)"
)
ipython.run_cell_magic("spark", "", codigo_remoto_estudo)


## Disponibilizar o ZIP no JupyterLab

A última célula transfere também pacotes parciais e de diagnóstico. Em caso de falha na transferência, o caminho HDFS e o SHA-256 ficam disponíveis na saída anterior. Para repetir apenas o download, execute esta célula novamente.


In [ ]:
if baixar_zip_jupyterlab:
    import hashlib
    import os
    import re
    from tempfile import TemporaryDirectory
    from urllib.parse import urlsplit

    from bbmagic import Hdfs

    recibo_gfp_local = None
    get_ipython().run_cell_magic(
        "spark", "-o recibo_gfp_local -n 1",
        "recibo_gfp_local = spark.createDataFrame([recibo_zip_gfp], "
        "'execucao string, caminho string, sha256 string')",
    )
    if recibo_gfp_local is None or len(recibo_gfp_local) != 1:
        raise RuntimeError("Recibo indisponível no JupyterLab. Consulte o caminho publicado no HDFS.")
    recibo = recibo_gfp_local.iloc[0]
    if recibo["execucao"] != id_execucao_gfp or not re.fullmatch(r"[0-9a-f]{64}", str(recibo["sha256"])):
        raise RuntimeError("A execução atual não possui um ZIP validado para download.")
    uri_zip = urlsplit(recibo["caminho"])
    if uri_zip.scheme != "hdfs" or not uri_zip.netloc or not uri_zip.path.endswith(".zip"):
        raise ValueError("Caminho do ZIP inválido no recibo.")
    destino_zip = Path.cwd() / f"{id_execucao_gfp}_{Path(uri_zip.path).name}"
    if destino_zip.exists():
        raise FileExistsError(destino_zip.name)
    with TemporaryDirectory(dir=destino_zip.parent) as pasta_download:
        temporario = Path(pasta_download) / "pacote.zip"
        try:
            Hdfs(gerenciador_local.keytab, cluster=cluster_bbmagic).download(
                uri_zip.path, str(temporario), overwrite=False,
            )
        except Exception as erro:
            raise RuntimeError(f"Falha no download BBMagic ({type(erro).__name__}). O ZIP permanece no HDFS.") from None
        os.chmod(temporario, 0o600)
        digest = hashlib.sha256()
        with temporario.open("rb") as entrada:
            for bloco in iter(lambda: entrada.read(262144), b""):
                digest.update(bloco)
        if digest.hexdigest() != recibo["sha256"]:
            raise RuntimeError("SHA-256 divergente no download. O original permanece no HDFS.")
        os.link(temporario, destino_zip)  # Publicação exclusiva no mesmo filesystem do JupyterLab.
    print("ZIP no JupyterLab com SHA-256 conferido", destino_zip)
